In [1]:
import sys
from pathlib import Path

sys.path.append(
    str(Path.cwd().parent / "src")
)

from extraction.gee import initialize
from extraction.sites import get_site
from extraction.products import get_product
from extraction.extractor import (
    _create_region,
    _load_collection,
    _reduce_image,
    _extract_collection,
    _to_dataframe,
    extract_timeseries,
)

In [2]:
initialize()

✓ Earth Engine initialized successfully.


In [3]:
site = get_site("BFT")
product = get_product("MOD16A2GF")

site, product

(Site(id='BFT', latitude=21.86, longitude=77.42, elevation=507, buffer_m=700),
 ETProduct(name='MOD16A2GF', product_type='Remote Sensing', gee_collection='MODIS/061/MOD16A2GF', band='ET', scale=500, temporal_resolution='8-day', scale_factor=0.1, units='kg m-2 / 8-day', provider='NASA'))

In [4]:
region = _create_region(site)
region

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Geometry.buffer",
    "arguments": {
      "distance": {
        "constantValue": 700
      },
      "geometry": {
        "functionInvocationValue": {
          "functionName": "GeometryConstructors.Point",
          "arguments": {
            "coordinates": {
              "constantValue": [
                77.42,
                21.86
              ]
            }
          }
        }
      }
    }
  }
})

In [5]:
collection = _load_collection(
    product,
    "2016-01-01",
    "2016-12-31",
)
collection.size().getInfo()

46

In [6]:
first_image = collection.first()
feature = _reduce_image(
    first_image,
    region,
    product,
)
feature.getInfo()

{'type': 'Feature',
 'geometry': None,
 'properties': {'ET': 37.936005171299286, 'date': '2016-01-01'}}

In [7]:
features = _extract_collection(
    collection,
    region,
    product,
)
features.size().getInfo()

46

In [8]:
df = _to_dataframe(
    features,
    product,
)
df.head()

,Date,DoY,ET
0,2016-01-01,1,3.793601
1,2016-01-09,9,3.947705
2,2016-01-17,17,4.724370
3,2016-01-25,25,2.116354
4,2016-02-02,33,2.866710


In [9]:
df = extract_timeseries(
    site,
    product,
    "2016-01-01",
    "2016-12-31",
)
df.head()

,Date,DoY,ET
0,2016-01-01,1,3.793601
1,2016-01-09,9,3.947705
2,2016-01-17,17,4.724370
3,2016-01-25,25,2.116354
4,2016-02-02,33,2.866710
